# Планирование следующего эксперимента

1. Рассчитаю размер выборки для нескольких минимальных эффектов.
2. Проведу A/A-симуляции, чтобы оценить частоту ложных срабатываний.
3. Проверю, помогает ли история покупок повысить точность оценки с помощью CUPED.

## Размер выборки

Основная метрика — средние расходы на клиента за две недели. Рассмотрю будущий тест с тремя равными группами: контроль и два варианта рассылки. Для каждого сравнения с контролем запланирую мощность 80%.

In [3]:
from math import ceil
from pathlib import Path

import pandas as pd
from statsmodels.stats.power import TTestIndPower

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

df = pd.read_csv(root / "data" / "hillstrom.csv")

spend_std = df.groupby("segment")["spend"].std()
display(spend_std.round(2).to_frame("Стандартное отклонение, $"))

,"Стандартное отклонение, $"
segment,
Mens E-Mail,17.75
No E-Mail,11.59
Womens E-Mail,15.12


Для планирования возьмём наибольшее стандартное отклонение среди трёх групп Hillstrom. Так расчёт будет консервативнее, чем при использовании меньшего исторического разброса. Это предположение о будущем тесте.

MDE — минимальный эффект, который мы хотим обнаруживать с заданной мощностью. Например, при MDE $0,20 и мощности 80%, если истинный эффект равен $0,20 на клиента, тест обнаружит его примерно в 80% повторений при заданных условиях.

У нас два сравнения с общим контролем. Для планирования используем консервативный порог 0,05 / 2 = 0,025 на сравнение. solve_power принимает эффект в единицах стандартного отклонения, поэтому делим MDE в долларах на planning_std

In [4]:
planning_std = spend_std.max()
mde_values = [0.10, 0.20, 0.30, 0.50]
power_model = TTestIndPower()
sample_sizes = []

for mde in mde_values:
    clients_per_group = ceil(
        power_model.solve_power(
            effect_size=mde / planning_std,
            alpha=0.025,
            power=0.80,
            ratio=1,
            alternative="two-sided",
        )
    )

    sample_sizes.append({
        "MDE, $ на клиента": mde,
        "Клиентов на группу": clients_per_group,
        "Всего клиентов": clients_per_group * 3,
    })

sample_size_table = pd.DataFrame(sample_sizes)
display(sample_size_table)

,"MDE, $ на клиента",Клиентов на группу,Всего клиентов
0,0.1,599222,1797666
1,0.2,149807,449421
2,0.3,66582,199746
3,0.5,23971,71913


Чтобы обнаруживать прирост $0,50 на клиента с мощностью 80%, по расчёту нужно 23 971 клиент на группу, или 71 913 клиентов всего. Если уменьшить MDE с $0,20 до $0,10, нужная выборка вырастет примерно в четыре раза — с 149 807 до 599 222 клиентов на группу.

In [5]:
# Теперь решим обратную задачу: если у нас снова 64 000 клиентов, какой MDE получится?
clients_per_group = len(df) // 3

mde_standardized = power_model.solve_power(
    effect_size=None,
    nobs1=clients_per_group,
    alpha=0.025,
    power=0.80,
    ratio=1,
    alternative="two-sided",
)

mde_usd = mde_standardized * planning_std
print(f"MDE при {clients_per_group:,} клиентах на группу: ${mde_usd:.2f}")

MDE при 21,333 клиентах на группу: $0.53


## A/A-симуляция

Случайно разделю клиентов контрольной группы на две половины и сравню их средние расходы двусторонним критерием Уэлча. Повторю разбиение 2 000 раз. Для каждого разбиения проверю одну гипотезу при уровне значимости 0,05. Доля разбиений с p-value ниже 0,05 покажет частоту ложных срабатываний в этой симуляции.

In [6]:
import numpy as np
from scipy.stats import ttest_ind

control_spend = df.loc[df["segment"] == "No E-Mail", "spend"].to_numpy()

rng = np.random.default_rng(42)
n_simulations = 2_000
alpha_aa = 0.05
aa_p_values = []

for _ in range(n_simulations):
    shuffled_spend = rng.permutation(control_spend)
    group_a, group_a_prime = np.array_split(shuffled_spend, 2)

    result = ttest_ind(
        group_a,
        group_a_prime,
        equal_var=False,
        alternative="two-sided",
    )
    aa_p_values.append(result.pvalue)

aa_p_values = np.array(aa_p_values)
false_positives = (aa_p_values < alpha_aa).sum()
false_positive_rate = false_positives / n_simulations

print(f"Клиентов в каждой половине: {len(control_spend) // 2:,}")
print(f"Ложных срабатываний: {false_positives} из {n_simulations}")
print(f"Частота ложных срабатываний: {false_positive_rate:.2%}")

Клиентов в каждой половине: 10,653
Ложных срабатываний: 92 из 2000
Частота ложных срабатываний: 4.60%


### Вывод по A/A-симуляции

В 2 000 случайных разбиениях контрольной группы критерий Уэлча выявил значимое различие 92 раза (4,60%). Наблюдаемая частота ложных срабатываний близка к заданному уровню значимости 5%.

Теперь — CUPED. Сравним две оценки эффекта каждой рассылки: обычную разницу средних и оценку с поправкой на history — расходы клиента до рассылки. Если прошлые расходы помогают объяснить расходы за две недели эксперимента, поправка может уменьшить стандартную ошибку и сузить интервал. Здесь используем регрессионный вариант корректировки.

In [7]:
import statsmodels.formula.api as smf

cuped_rows = []

for group in ["Mens E-Mail", "Womens E-Mail"]:
    comparison = df.loc[
        df["segment"].isin(["No E-Mail", group]),
        ["segment", "spend", "history"],
    ].copy()

    comparison["treated"] = comparison["segment"].eq(group).astype(int)

    formulas = {
        "Без поправки": "spend ~ treated",
        "С учётом history": "spend ~ treated + history",
    }

    for method, formula in formulas.items():
        model = smf.ols(formula, data=comparison, missing="raise").fit(
            cov_type="HC3"
        )
        ci_low, ci_high = model.conf_int().loc["treated"]

        cuped_rows.append({
            "Группа": group,
            "Метод": method,
            "Эффект, $": model.params["treated"],
            "Стандартная ошибка": model.bse["treated"],
            "Нижняя граница 95% ДИ": ci_low,
            "Верхняя граница 95% ДИ": ci_high,
        })

cuped_results = pd.DataFrame(cuped_rows)
display(cuped_results.round(4))

,Группа,Метод,"Эффект, $",Стандартная ошибка,Нижняя граница 95% ДИ,Верхняя граница 95% ДИ
0,Mens E-Mail,Без поправки,0.7698,0.1452,0.4851,1.0545
1,Mens E-Mail,С учётом history,0.7674,0.1452,0.4830,1.0519
2,Womens E-Mail,Без поправки,0.4244,0.1303,0.1690,0.6799
3,Womens E-Mail,С учётом history,0.4227,0.1302,0.1675,0.6780


### Вывод по CUPED

Поправка на исторические расходы клиента (`history`) практически не изменила оценки эффекта и ширину доверительных интервалов. В обоих сравнениях с контролем интервалы остались целиком выше нуля.

На данных Hillstrom исторические расходы в этой модели дали минимальный выигрыш в точности. Для итогового отчёта использую оценки из основного A/B-анализа.